In [8]:
import pyspark as pyspark
from pyspark.sql import SparkSession


In [9]:
pyspark.__file__

'/home/sultan_ali/spark/spark-3.5.9-bin-hadoop3/python/pyspark/__init__.py'

In [10]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("test") \
    .getOrCreate()

26/11/23 06:14:21 WARN Utils: Your hostname, LAPTOP-FH4AVPBT resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/11/23 06:14:21 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/23 06:14:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/23 06:14:27 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [11]:
df = spark.read \
    .option("header","true") \
    .csv('../data/fhvhv_tripdata_2021-01.csv.gz')
df.show()

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|           HV0003|              B02682|2021-01-01 00:33:44|2021-01-01 00:49:07|         230|         166|   NULL|
|           HV0003|              B02682|2021-01-01 00:55:19|2021-01-01 01:18:21|         152|         167|   NULL|
|           HV0003|              B02764|2021-01-01 00:23:56|2021-01-01 00:38:05|         233|         142|   NULL|
|           HV0003|              B02764|2021-01-01 00:42:51|2021-01-01 00:45:50|         142|         143|   NULL|
|           HV0003|              B02764|2021-01-01 00:48:14|2021-01-01 01:08:42|         143|          78|   NULL|
|           HV0005|              B02510|2021-01-01 00:06:59|2021-01-01 00:43:01|

In [14]:
df.head(5)

[Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', pickup_datetime='2021-01-01 00:33:44', dropoff_datetime='2021-01-01 00:49:07', PULocationID='230', DOLocationID='166', SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', pickup_datetime='2021-01-01 00:55:19', dropoff_datetime='2021-01-01 01:18:21', PULocationID='152', DOLocationID='167', SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime='2021-01-01 00:23:56', dropoff_datetime='2021-01-01 00:38:05', PULocationID='233', DOLocationID='142', SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime='2021-01-01 00:42:51', dropoff_datetime='2021-01-01 00:45:50', PULocationID='142', DOLocationID='143', SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime='2021-01-01 00:48:14', dropoff_datetime='2021-01-01 01:08:42', PULocationID='143', DOLocationID='78', SR_Flag=None)]

### 1. Fix cols data types

In [15]:
df.schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', StringType(), True), StructField('DOLocationID', StringType(), True), StructField('SR_Flag', StringType(), True)])

In [17]:
from pyspark.sql import types

In [18]:
schema = types.StructType([
    types.StructField('hvfhs_license_num', types.StringType(), True), 
    types.StructField('dispatching_base_num', types.StringType(), True), 
    types.StructField('pickup_datetime', types.TimestampType(), True), 
    types.StructField('dropoff_datetime', types.TimestampType(), True), 
    types.StructField('PULocationID', types.IntegerType(), True), 
    types.StructField('DOLocationID', types.IntegerType(), True), 
    types.StructField('SR_Flag', types.StringType(), True)
])

In [19]:
df = spark.read \
    .option("header","true") \
    .schema(schema) \
    .csv('../data/fhvhv_tripdata_2021-01.csv.gz')
df.show()

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|           HV0003|              B02682|2021-01-01 00:33:44|2021-01-01 00:49:07|         230|         166|   NULL|
|           HV0003|              B02682|2021-01-01 00:55:19|2021-01-01 01:18:21|         152|         167|   NULL|
|           HV0003|              B02764|2021-01-01 00:23:56|2021-01-01 00:38:05|         233|         142|   NULL|
|           HV0003|              B02764|2021-01-01 00:42:51|2021-01-01 00:45:50|         142|         143|   NULL|
|           HV0003|              B02764|2021-01-01 00:48:14|2021-01-01 01:08:42|         143|          78|   NULL|
|           HV0005|              B02510|2021-01-01 00:06:59|2021-01-01 00:43:01|

In [23]:
df.schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', TimestampType(), True), StructField('dropoff_datetime', TimestampType(), True), StructField('PULocationID', IntegerType(), True), StructField('DOLocationID', IntegerType(), True), StructField('SR_Flag', StringType(), True)])

In [24]:
!ls

Untitled.ipynb


In [25]:
df = df.repartition(24)

In [26]:
df.write.parquet('fhvhv/2021/01')

26/11/23 08:33:02 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/11/23 08:33:02 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/11/23 08:33:02 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/11/23 08:33:02 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/11/23 08:33:02 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/11/23 08:33:02 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/11/23 08:33:02 WARN MemoryManager: Total allocation exceeds 95.

In [27]:
df_parq = spark.read.parquet('fhvhv/2021/01')

In [36]:
df_parq

DataFrame[hvfhs_license_num: string, dispatching_base_num: string, pickup_datetime: timestamp, dropoff_datetime: timestamp, PULocationID: int, DOLocationID: int, SR_Flag: string]

In [37]:
df_parq.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- SR_Flag: string (nullable = true)



In [38]:
display(df_parq.head(5))

[Row(hvfhs_license_num='HV0005', dispatching_base_num='B02510', pickup_datetime=datetime.datetime(2021, 1, 10, 17, 40, 1), dropoff_datetime=datetime.datetime(2021, 1, 10, 17, 48, 10), PULocationID=97, DOLocationID=25, SR_Flag=None),
 Row(hvfhs_license_num='HV0005', dispatching_base_num='B02510', pickup_datetime=datetime.datetime(2021, 1, 8, 18, 19, 44), dropoff_datetime=datetime.datetime(2021, 1, 8, 18, 55, 57), PULocationID=138, DOLocationID=265, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02876', pickup_datetime=datetime.datetime(2021, 1, 1, 16, 47, 20), dropoff_datetime=datetime.datetime(2021, 1, 1, 16, 58, 28), PULocationID=50, DOLocationID=163, SR_Flag=None),
 Row(hvfhs_license_num='HV0005', dispatching_base_num='B02510', pickup_datetime=datetime.datetime(2021, 1, 15, 17, 50, 8), dropoff_datetime=datetime.datetime(2021, 1, 15, 18, 7, 24), PULocationID=163, DOLocationID=79, SR_Flag=None),
 Row(hvfhs_license_num='HV0005', dispatching_base_num='B02510', pic

In [39]:
df_parq.columns

['hvfhs_license_num',
 'dispatching_base_num',
 'pickup_datetime',
 'dropoff_datetime',
 'PULocationID',
 'DOLocationID',
 'SR_Flag']

In [40]:
df_parq.count()

11908468

In [42]:
df_parq

DataFrame[hvfhs_license_num: string, dispatching_base_num: string, pickup_datetime: timestamp, dropoff_datetime: timestamp, PULocationID: int, DOLocationID: int, SR_Flag: string]

###  2. Spark Df Basics

####  Select colums

In [44]:
df_parq.select("hvfhs_license_num", "DOLocationID") \
    .show(3)

+-----------------+------------+
|hvfhs_license_num|DOLocationID|
+-----------------+------------+
|           HV0005|          25|
|           HV0005|         265|
|           HV0003|         163|
+-----------------+------------+
only showing top 3 rows



In [49]:
df_parq.select("hvfhs_license_num", "DOLocationID") \
    .schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('DOLocationID', IntegerType(), True)])

####  Select/Filter Rows

In [51]:
df_parq \
    .filter(
        (df_parq.DOLocationID <= 100) &
        (df_parq.DOLocationID >= 90)
    ) \
    .show()

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|           HV0003|              B02765|2021-01-05 02:00:14|2021-01-05 02:19:39|          48|          95|   NULL|
|           HV0005|              B02510|2021-01-16 00:28:56|2021-01-16 01:27:38|          92|          92|   NULL|
|           HV0003|              B02765|2021-01-16 15:59:36|2021-01-16 16:26:51|         113|          97|   NULL|
|           HV0003|              B02884|2021-01-13 17:30:24|2021-01-13 17:50:02|         241|          94|   NULL|
|           HV0005|              B02510|2021-01-16 20:13:09|2021-01-16 20:31:14|         210|          91|   NULL|
|           HV0003|              B02765|2021-01-24 21:45:57|2021-01-24 21:49:53|

In [52]:
df_parq \
    .select("hvfhs_license_num", "dispatching_base_num", "DOLocationID") \
    .filter(
        (df_parq.DOLocationID <= 100) &
        (df_parq.DOLocationID >= 90)
    ) \
    .show()

+-----------------+--------------------+------------+
|hvfhs_license_num|dispatching_base_num|DOLocationID|
+-----------------+--------------------+------------+
|           HV0003|              B02765|          95|
|           HV0005|              B02510|          92|
|           HV0003|              B02765|          97|
|           HV0003|              B02884|          94|
|           HV0005|              B02510|          91|
|           HV0003|              B02765|          96|
|           HV0003|              B02835|          92|
|           HV0005|              B02510|          95|
|           HV0003|              B02835|          92|
|           HV0005|              B02510|         100|
|           HV0003|              B02882|          90|
|           HV0003|              B02888|          92|
|           HV0003|              B02875|          90|
|           HV0003|              B02875|          91|
|           HV0005|              B02510|          97|
|           HV0003|         

In [53]:
df_parq \
    .select("hvfhs_license_num", "dispatching_base_num", "DOLocationID") \
    .filter(df_parq.DOLocationID.between(100,90)) \
    .show()

+-----------------+--------------------+------------+
|hvfhs_license_num|dispatching_base_num|DOLocationID|
+-----------------+--------------------+------------+
+-----------------+--------------------+------------+

